In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="aIuJiuSXzAmwEQFYVFMD")
project = rf.workspace("hazimsensoryai").project("generalstation")
version = project.version(1)
dataset = version.download("folder")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to GeneralStation-1 in folder:: 100%|██████████| 5462/5462 [00:00<00:00, 8347.63it/s]


In [2]:
import os

for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, "").count(os.sep)

    if level > 2:
        continue

    print(root)

/content/GeneralStation-1
/content/GeneralStation-1/test
/content/GeneralStation-1/test/1
/content/GeneralStation-1/test/0
/content/GeneralStation-1/valid
/content/GeneralStation-1/valid/1
/content/GeneralStation-1/valid/0
/content/GeneralStation-1/train
/content/GeneralStation-1/train/1
/content/GeneralStation-1/train/0


In [3]:
import os

dataset_path = "/content/GeneralStation-1"

for split in ["train", "valid", "test"]:
    print(f"\n{split.upper()}")

    for label in ["0", "1"]:
        folder = os.path.join(
            dataset_path,
            split,
            label
        )

        count = len([
            f for f in os.listdir(folder)
            if os.path.isfile(os.path.join(folder, f))
        ])

        class_name = "NOCUP" if label == "0" else "CUP"

        print(f"  {label} ({class_name}): {count}")


TRAIN
  0 (NOCUP): 3390
  1 (CUP): 1359

VALID
  0 (NOCUP): 324
  1 (CUP): 135

TEST
  0 (NOCUP): 164
  1 (CUP): 88


In [12]:
DATASET_PATH = "/content/GeneralStation-1"
RESULTS_DIR = "/content/GeneralStation-1_results"

RUN_HOG_SVM = True
RUN_HOG_RF = False
RUN_HOG_XGBOOST = False
RUN_LBP_SVM = False
RUN_LBP_RF = False
RUN_HOG_LBP_SVM = True
RUN_PCA_SVM = True
RUN_RAW_PIXELS_SVM = True
RUN_RESNET18 = True
RUN_MOBILENET = True

CNN_EPOCHS = 15
CNN_BATCH_SIZE = 32
CNN_LR = 1e-4
CNN_PATIENCE = 4
SEED = 42

In [5]:
!pip install -q scikit-image scikit-learn matplotlib seaborn pyyaml joblib

In [6]:
import os, random, copy, time, yaml, joblib
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog, local_binary_pattern
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from xgboost import XGBClassifier

random.seed(SEED)
np.random.seed(SEED)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "confusion_matrices"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "models"), exist_ok=True)

CLASS_NAMES = {
    0: "NOCUP",
    1: "CUP"
}

In [7]:
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VALID_PATH = os.path.join(DATASET_PATH, "valid")
TEST_PATH = os.path.join(DATASET_PATH, "test")

def load_images(folder, size=(224, 224)):
    images, labels, paths = [], [], []
    for label in [0, 1]:
        class_folder = os.path.join(folder, str(label))
        if not os.path.isdir(class_folder):
            raise FileNotFoundError(class_folder)
        for filename in sorted(os.listdir(class_folder)):
            path = os.path.join(class_folder, filename)
            if not os.path.isfile(path):
                continue
            image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            image = cv2.resize(image, size)
            images.append(image)
            labels.append(label)
            paths.append(path)
    return np.asarray(images), np.asarray(labels), paths

X_train_img, y_train, train_paths = load_images(TRAIN_PATH)
X_valid_img, y_valid, valid_paths = load_images(VALID_PATH)
X_test_img, y_test, test_paths = load_images(TEST_PATH)

print("Train:", len(y_train))
print("Valid:", len(y_valid))
print("Test :", len(y_test))
print("Classes:", CLASS_NAMES)


Train: 4749
Valid: 459
Test : 252
Classes: {0: 'NOCUP', 1: 'CUP'}


In [8]:
dataset_yaml = {
    "path": DATASET_PATH,
    "train": "train",
    "val": "valid",
    "test": "test",
    "names": {0: "NOCUP", 1: "CUP"}
}
yaml_path = "/content/GeneralStation-1_dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False)
print(open(yaml_path).read())


path: /content/GeneralStation-1
train: train
val: valid
test: test
names:
  0: NOCUP
  1: CUP



In [9]:
def extract_hog(images):
    return np.asarray([
        hog(im, orientations=9, pixels_per_cell=(8, 8),
            cells_per_block=(2, 2), block_norm="L2-Hys")
        for im in images
    ], dtype=np.float32)

def extract_lbp(images, points=24, radius=3, grid=(4, 4)):
    features = []
    n_bins = points + 2
    gh, gw = grid
    for im in images:
        feats = []
        h, w = im.shape
        for gy in range(gh):
            for gx in range(gw):
                y0, y1 = int(gy*h/gh), int((gy+1)*h/gh)
                x0, x1 = int(gx*w/gw), int((gx+1)*w/gw)
                cell = im[y0:y1, x0:x1]
                lbp = local_binary_pattern(cell, points, radius, method="uniform")
                hist, _ = np.histogram(
                    lbp.ravel(), bins=np.arange(0, n_bins+1),
                    range=(0, n_bins), density=True
                )
                feats.extend(hist)
        features.append(feats)
    return np.asarray(features, dtype=np.float32)

HOG_train = extract_hog(X_train_img)
HOG_valid = extract_hog(X_valid_img)
HOG_test = extract_hog(X_test_img)

LBP_train = extract_lbp(X_train_img)
LBP_valid = extract_lbp(X_valid_img)
LBP_test = extract_lbp(X_test_img)

RAW_train = X_train_img.reshape(len(X_train_img), -1).astype(np.float32) / 255.0
RAW_valid = X_valid_img.reshape(len(X_valid_img), -1).astype(np.float32) / 255.0
RAW_test = X_test_img.reshape(len(X_test_img), -1).astype(np.float32) / 255.0

print("HOG:", HOG_train.shape)
print("LBP:", LBP_train.shape)
print("RAW:", RAW_train.shape)


HOG: (4749, 26244)
LBP: (4749, 416)
RAW: (4749, 50176)


In [10]:
# ============================================================
# MODEL EVALUATION + CHECKPOINT HELPERS
# ============================================================

all_results = []
all_predictions = {}
trained_models = {}


def evaluate_model(name, model, Xtr, ytr, Xv, yv, Xt, yt):
    """
    Train and evaluate one classical ML model.
    """

    print(f"\n{'=' * 60}")
    print(f"Training: {name}")
    print(f"{'=' * 60}")

    start = time.time()

    # Train
    model.fit(Xtr, ytr)

    # Validation prediction
    val_pred = model.predict(Xv)

    # Test prediction
    test_pred = model.predict(Xt)

    # Confusion matrix
    cm = confusion_matrix(
        yt,
        test_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    # Metrics
    row = {
        "Method": name,

        "Validation Balanced Accuracy (%)":
            balanced_accuracy_score(yv, val_pred) * 100,

        "Test Accuracy (%)":
            accuracy_score(yt, test_pred) * 100,

        "Test Balanced Accuracy (%)":
            balanced_accuracy_score(yt, test_pred) * 100,

        "CUP Precision (%)":
            precision_score(
                yt,
                test_pred,
                pos_label=1,
                zero_division=0
            ) * 100,

        "CUP Recall (%)":
            recall_score(
                yt,
                test_pred,
                pos_label=1,
                zero_division=0
            ) * 100,

        "NOCUP Specificity (%)":
            (tn / (tn + fp) * 100)
            if (tn + fp) else 0,

        "CUP F1 (%)":
            f1_score(
                yt,
                test_pred,
                pos_label=1,
                zero_division=0
            ) * 100,

        "Runtime (s)":
            time.time() - start
    }

    # Store results in memory
    all_results.append(row)
    all_predictions[name] = test_pred
    trained_models[name] = model

    # Display results
    print(f"\n{name}")
    print(pd.Series(row))

    print("\nConfusion Matrix:")
    print(cm)

    return cm

In [14]:
# ================= PER-METHOD CHECKPOINT / ZIP HELPERS =================

import json
import shutil
from pathlib import Path

# Temporary checkpoint folder in Colab
METHOD_OUTPUT_DIR = "/content/method_checkpoints"

os.makedirs(METHOD_OUTPUT_DIR, exist_ok=True)


def _safe_name(name):
    return "".join(
        c if c.isalnum() or c in "+-_"
        else "_"
        for c in name
    )


def method_zip_path(name):
    return f"/content/{_safe_name(name)}.zip"

def save_method_checkpoint(
    name,
    row,
    model=None,
    predictions=None,
    confusion=None,
    history=None,
):
    """Save one completed method immediately as /content/<Method>.zip."""
    safe = _safe_name(name)
    work_dir = os.path.join(METHOD_OUTPUT_DIR, safe)
    model_dir = os.path.join(work_dir, "model")
    os.makedirs(model_dir, exist_ok=True)

    # Save metrics
    with open(os.path.join(work_dir, "metrics.json"), "w") as f:
        json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                   for k, v in row.items()}, f, indent=2)

    pd.DataFrame([row]).to_csv(os.path.join(work_dir, "metrics.csv"), index=False)

    # Save predictions
    if predictions is not None:
        np.save(os.path.join(work_dir, "test_predictions.npy"), np.asarray(predictions))

    # Save confusion matrix
    if confusion is not None:
        np.save(os.path.join(work_dir, "confusion_matrix.npy"), np.asarray(confusion))

    # Save CNN history if applicable
    if history is not None:
        with open(os.path.join(work_dir, "history.json"), "w") as f:
            json.dump({k: [float(x) for x in v] for k, v in history.items()}, f, indent=2)

    # Save model
    if model is not None:
        if name in ["ResNet18", "MobileNetV3-Small"]:
            torch.save(model.state_dict(), os.path.join(model_dir, safe + ".pth"))
        else:
            joblib.dump(model, os.path.join(model_dir, safe + ".pkl"))

    # ZIP the completed method immediately.
    zip_base = f"/content/{safe}"
    zip_path = shutil.make_archive(zip_base, "zip", work_dir)

    print(f"\nCHECKPOINT SAVED: {zip_path}")
    print("This method is now safe even if the runtime crashes.")
    return zip_path

def checkpoint_exists(name):
    return os.path.exists(method_zip_path(name))


In [ ]:
if RUN_HOG_SVM:
    name = "HOG + SVM"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        cm = evaluate_model(
            name,
            SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=SEED),
            HOG_train, y_train, HOG_valid, y_valid, HOG_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )



Training: HOG + SVM


In [22]:
if RUN_HOG_RF:
    name = "HOG + Random Forest"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        cm = evaluate_model(
            name,
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            ),
            HOG_train, y_train, HOG_valid, y_valid, HOG_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )



Training: HOG + Random Forest

HOG + Random Forest
Method                              HOG + Random Forest
Validation Balanced Accuracy (%)              84.444444
Test Accuracy (%)                             91.666667
Test Balanced Accuracy (%)                    88.068182
CUP Precision (%)                                 100.0
CUP Recall (%)                                76.136364
NOCUP Specificity (%)                             100.0
CUP F1 (%)                                    86.451613
Runtime (s)                                  199.018124
dtype: object

Confusion Matrix:
[[164   0]
 [ 21  67]]

CHECKPOINT SAVED: /content/HOG_+_Random_Forest.zip
This method is now safe even if the runtime crashes.


In [23]:
if RUN_HOG_XGBOOST:
    name = "HOG + XGBoost"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        cm = evaluate_model(
            name,
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=SEED,
                n_jobs=-1
            ),
            HOG_train, y_train, HOG_valid, y_valid, HOG_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )



Training: HOG + XGBoost

HOG + XGBoost
Method                              HOG + XGBoost
Validation Balanced Accuracy (%)         90.95679
Test Accuracy (%)                       96.031746
Test Balanced Accuracy (%)              94.844789
CUP Precision (%)                       97.560976
CUP Recall (%)                          90.909091
NOCUP Specificity (%)                   98.780488
CUP F1 (%)                              94.117647
Runtime (s)                           1145.408681
dtype: object

Confusion Matrix:
[[162   2]
 [  8  80]]

CHECKPOINT SAVED: /content/HOG_+_XGBoost.zip
This method is now safe even if the runtime crashes.


In [24]:
if RUN_LBP_SVM:
    name = "LBP + SVM"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        cm = evaluate_model(
            name,
            SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=SEED),
            LBP_train, y_train, LBP_valid, y_valid, LBP_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )



Training: LBP + SVM

LBP + SVM
Method                              LBP + SVM
Validation Balanced Accuracy (%)    89.969136
Test Accuracy (%)                   92.460317
Test Balanced Accuracy (%)          90.521064
CUP Precision (%)                   93.670886
CUP Recall (%)                      84.090909
NOCUP Specificity (%)                96.95122
CUP F1 (%)                          88.622754
Runtime (s)                         11.168905
dtype: object

Confusion Matrix:
[[159   5]
 [ 14  74]]

CHECKPOINT SAVED: /content/LBP_+_SVM.zip
This method is now safe even if the runtime crashes.


In [25]:
if RUN_LBP_RF:
    name = "LBP + Random Forest"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        cm = evaluate_model(
            name,
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            ),
            LBP_train, y_train, LBP_valid, y_valid, LBP_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )



Training: LBP + Random Forest

LBP + Random Forest
Method                              LBP + Random Forest
Validation Balanced Accuracy (%)              84.722222
Test Accuracy (%)                             89.285714
Test Balanced Accuracy (%)                    84.922395
CUP Precision (%)                             98.412698
CUP Recall (%)                                70.454545
NOCUP Specificity (%)                         99.390244
CUP F1 (%)                                    82.119205
Runtime (s)                                   16.883386
dtype: object

Confusion Matrix:
[[163   1]
 [ 26  62]]

CHECKPOINT SAVED: /content/LBP_+_Random_Forest.zip
This method is now safe even if the runtime crashes.


In [ ]:
if RUN_HOG_LBP_SVM:
    name = "HOG + LBP + SVM"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        Xtr = np.hstack([HOG_train, LBP_train])
        Xv = np.hstack([HOG_valid, LBP_valid])
        Xt = np.hstack([HOG_test, LBP_test])

        cm = evaluate_model(
            name,
            SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=SEED),
            Xtr, y_train, Xv, y_valid, Xt, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )


In [ ]:
if RUN_RAW_PIXELS_SVM:
    name = "Raw Pixels + SVM"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        raw_svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=SEED))
        ])

        cm = evaluate_model(
            name,
            raw_svm,
            RAW_train, y_train, RAW_valid, y_valid, RAW_test, y_test
        )
        save_method_checkpoint(
            name,
            all_results[-1],
            model=trained_models[name],
            predictions=all_predictions[name],
            confusion=cm
        )


In [ ]:
# ================= CNN SETUP =================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

cnn_train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485]*3, [0.229]*3)
])

cnn_eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485]*3, [0.229]*3)
])

cnn_train = datasets.ImageFolder(TRAIN_PATH, transform=cnn_train_tf)
cnn_valid = datasets.ImageFolder(VALID_PATH, transform=cnn_eval_tf)
cnn_test = datasets.ImageFolder(TEST_PATH, transform=cnn_eval_tf)

train_loader = DataLoader(cnn_train, batch_size=CNN_BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(cnn_valid, batch_size=CNN_BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(cnn_test, batch_size=CNN_BATCH_SIZE, shuffle=False, num_workers=2)

print("CNN class mapping:", cnn_train.class_to_idx)


In [ ]:
# ================= CNN FUNCTIONS =================
def build_cnn(kind):
    if kind == "ResNet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 2)
    else:
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    return model.to(device)

def run_cnn(name):
    model = build_cnn(name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CNN_LR)

    best_state = copy.deepcopy(model.state_dict())
    best_val = -1
    wait = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()

    for epoch in range(CNN_EPOCHS):
        model.train()
        total = correct = running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (output.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        total = correct = val_loss_sum = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                output = model(images)
                loss = criterion(output, labels)
                val_loss_sum += loss.item() * labels.size(0)
                correct += (output.argmax(1) == labels).sum().item()
                total += labels.size(0)

        val_loss = val_loss_sum / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"{name} | Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
              f"train {train_acc*100:.2f}% | val {val_acc*100:.2f}%")

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= CNN_PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    model.eval()
    predictions, labels_all = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            output = model(images.to(device))
            predictions.extend(output.argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())

    yt = np.asarray(labels_all)
    yp = np.asarray(predictions)
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    row = {
        "Method": name,
        "Validation Balanced Accuracy (%)": best_val * 100,
        "Test Accuracy (%)": accuracy_score(yt, yp) * 100,
        "Test Balanced Accuracy (%)": balanced_accuracy_score(yt, yp) * 100,
        "CUP Precision (%)": precision_score(yt, yp, pos_label=1, zero_division=0) * 100,
        "CUP Recall (%)": recall_score(yt, yp, pos_label=1, zero_division=0) * 100,
        "NOCUP Specificity (%)": (tn/(tn+fp)*100) if tn+fp else 0,
        "CUP F1 (%)": f1_score(yt, yp, pos_label=1, zero_division=0) * 100,
        "Runtime (s)": time.time() - start
    }

    all_results.append(row)
    all_predictions[name] = yp
    trained_models[name] = model
    cnn_histories[name] = history

    print("\n" + name + " FINAL TEST")
    print(pd.Series(row))
    print("Confusion Matrix:\n", cm)
    return row, model, yp, cm, history


In [ ]:
# ================= ResNet18 — ONE METHOD / ONE CELL =================
if RUN_RESNET18:
    name = "ResNet18"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        row, model, predictions, cm, history = run_cnn(name)
        save_method_checkpoint(
            name,
            row,
            model=model,
            predictions=predictions,
            confusion=cm,
            history=history
        )


In [ ]:
# ================= MobileNetV3-Small — ONE METHOD / ONE CELL =================
if RUN_MOBILENET:
    name = "MobileNetV3-Small"
    if checkpoint_exists(name):
        print(f"{method_zip_path(name)} already exists. Skipping {name}.")
    else:
        row, model, predictions, cm, history = run_cnn(name)
        save_method_checkpoint(
            name,
            row,
            model=model,
            predictions=predictions,
            confusion=cm,
            history=history
        )


In [ ]:
# ================= FINAL COMPARISON =================
# Rebuild the comparison from every completed method ZIP.
import zipfile

completed_rows = []

for zip_path in sorted(Path("/content").glob("*.zip")):
    # Only use method ZIPs created by this notebook.
    method_name = zip_path.stem
    if method_name not in {
        "HOG_+_SVM", "HOG_+_Random_Forest", "HOG_+_XGBoost",
        "LBP_+_SVM", "LBP_+_Random_Forest", "HOG_+_LBP_+_SVM",
        "Raw_Pixels_+_SVM", "ResNet18", "MobileNetV3-Small"
    }:
        continue

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            metrics_files = [n for n in z.namelist() if n.endswith("metrics.json")]
            if metrics_files:
                with z.open(metrics_files[0]) as f:
                    completed_rows.append(json.load(f))
    except Exception as e:
        print("Could not read:", zip_path, e)

results_df = pd.DataFrame(completed_rows)

if not results_df.empty:
    results_df = results_df.sort_values(
        "Test Balanced Accuracy (%)", ascending=False
    ).reset_index(drop=True)

    display(results_df.round(2))
    results_df.to_csv("/content/all_methods_comparison.csv", index=False)

    print("Best model:", results_df.iloc[0]["Method"])
    print("Comparison saved to: /content/all_methods_comparison.csv")
else:
    print("No completed method ZIPs found yet.")


## Checkpoint structure

Each completed method is saved immediately as its own ZIP in `/content/`.

Examples:
- `/content/HOG_+_SVM.zip`
- `/content/HOG_+_Random_Forest.zip`
- `/content/HOG_+_XGBoost.zip`
- `/content/LBP_+_SVM.zip`
- `/content/LBP_+_Random_Forest.zip`
- `/content/HOG_+_LBP_+_SVM.zip`
- `/content/Raw_Pixels_+_SVM.zip`
- `/content/ResNet18.zip`
- `/content/MobileNetV3-Small.zip`

If Colab crashes after a method finishes, its ZIP remains in `/content` for that runtime/session. Re-run the setup/data cells after reconnecting, then continue with the next method cell. If the ZIP still exists, the method cell automatically skips it.

PCA is intentionally excluded in this version, matching the current R&D setup.